In [13]:
workspace_id: str = '03cfd191-956e-489a-806c-4faf9e5cb1e5'
dataset_id: str  = '1a8c7ec1-3640-4776-bd4d-c1bcb9d3fb82'
enable_partition: bool = True
partitions_config: str = '[ { "table": "Sales", "first_date": "20200101", "partition_by": "Order Date", "interval": "QUARTER", "last_date": "TODAY", "intervals_to_refresh": "*" } ]'
enable_refresh: bool = True
tables_to_refresh: str = 'Customer,Sales'
partitions_to_refresh: str = ''
enable_referential_integrity: bool = True
refresh_mode: str = 'memory-optimized'
refresh_type: str = 'full'
refresh_commit_mode: str = 'transactional'
refresh_max_parallelism: int = 4
notebook_timeout: int = 7200

In [14]:
from datetime import datetime
from typing import Optional, Any, Dict
import os
import sys
import logging
import notebookutils
from io import StringIO
import uuid
import numpy as np

In [15]:
# Constants
DEFAULT_LOG_LEVEL = logging.DEBUG
FABTOOLKIT_VERSION = "1.0.0"
PARTITIONER_NOTEBOOK_NAME = "NB_PAR_PARTITIONER"
REFRESHER_NOTEBOOK_NAME = "NB_PAR_REFRESHER"
AVAILABLE_REFRESH_MODES = {"custom", "memory-optimized"}
AVAILABLE_REFRESH_TYPES = {"full", "clearValues", "calculate", "dataOnly", "automatic", "defragment"}
AVAILABLE_COMMIT_MODES = {"transactional", "partialBatch"}
DEFAULT_REFRESH_MODE = "custom"
DEFAULT_REFRESH_TYPE = "full"
DEFAULT_REFRESH_COMMIT_MODE = "transactional"
DEFAULT_REFRESH_MAX_PARALLELISM = 4
DEFAULT_NOTEBOOK_TIMEOUT = 7200

In [16]:
wheel_path = os.path.join(
    notebookutils.nbResPath,
    "builtin",
    f"fabtoolkit-{FABTOOLKIT_VERSION}-py3-none-any.whl"
)
%pip install {wheel_path}

Processing /synfs/nb_resource/builtin/fabtoolkit-1.0.0-py3-none-any.whl
fabtoolkit is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Note: you may need to restart the kernel to use updated packages.


In [17]:
from fabtoolkit.utils import (
    get_bounds_from_offset,
    generate_date_ranges,
    is_valid_text,
    validate_json,
    Constants
)
from fabtoolkit.log import ConsoleLogFormatter
import pandas as pd

In [18]:
logger = logging.getLogger("custom_refresh_orchestrator")
logger.setLevel(DEFAULT_LOG_LEVEL)
logger.propagate = False

if not logger.handlers:
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(ConsoleLogFormatter())
    console_handler.setLevel(DEFAULT_LOG_LEVEL)
    logger.addHandler(console_handler)

In [19]:
DATE_FORMAT = Constants.DATE_FORMAT

In [20]:
def validate_params(
        workspace_id: str,
        dataset_id: str,
        enable_partition: bool,
        partitions_config: str,
        enable_refresh: bool,
        tables_to_refresh: Optional[str],
        partitions_to_refresh: Optional[str],
        enable_referential_integrity: Optional[bool],
        refresh_mode: Optional[str],
        refresh_type: Optional[str],
        refresh_commit_mode: Optional[str],
        refresh_max_parallelism: Optional[int],
        notebook_timeout: Optional[int]
) -> Dict[str, Any]:
    """
    Validate input parameters.

    Args:
        workspace_id (str): The workspace identifier.
        dataset_id (str): The dataset identifier.
        enable_partition (bool): Flag to enable partitioning.
        partitions_config (str): JSON string for partitions configuration.
        enable_refresh (bool): Flag to enable refresh.
        tables_to_refresh (Optional[str]): Comma-separated table names to refresh.
        partitions_to_refresh (Optional[str]): JSON string with explicitly defined partitions to refresh.
        enable_referential_integrity (Optional[bool]): Flag to enable refreshing related tables to maintain referential integrity.
        refresh_mode (Optional[str]):  Refresh mode ('custom' or 'optimized') to use parameter-based settings or minimize memory usage peaks.
        refresh_type (Optional[str]): Refresh type used for the refresh operation.
        refresh_commit_mode (Optional[str]): Commit mode used for the refresh operation.
        refresh_max_parallelism (Optional[int]): Maximum parallelism used for the refresh operation.
        notebook_timeout (Optional[int]): Timeout for the notebook execution.

    Returns:
        Dict[str, Any]: Dictionary containing validated parameters.
    """

    try:
        uuid.UUID(workspace_id)
    except Exception as e:
        logger.error(f"Invalid workspace_id parameter: {str(e)}")
        raise ValueError("Invalid workspace_id parameter.") from e
    try:
        uuid.UUID(dataset_id)
    except Exception as e:
        logger.error(f"Invalid dataset_id parameter: {str(e)}")
        raise ValueError("Invalid dataset_id parameter.") from e

    if not isinstance(enable_partition, bool):
        logger.error("Invalid enable_partition parameter.")
        raise ValueError("Invalid enable_partition parameter.")
    if not isinstance(enable_refresh, bool):
        logger.error("Invalid enable_refresh parameter.")
        raise ValueError("Invalid enable_refresh parameter.")
    
    # Validate partitions_config JSON
    if (enable_partition or enable_refresh) and is_valid_text(partitions_config):
        columns = ["table", "first_date", "partition_by", "interval", "last_date", "intervals_to_refresh"]
        
        partitions_config_df: pd.DataFrame = validate_json(partitions_config, columns)
        
        # Parse last_date value if needed
        partitions_config_df["last_date"] = np.where(
            partitions_config_df["last_date"] == "TODAY",
            datetime.strftime(datetime.today(), DATE_FORMAT),
            partitions_config_df["last_date"]
        )
        partitions_config = partitions_config_df.to_json(orient="records")
        
    if enable_refresh:
        # Validate tables_to_refresh
        if is_valid_text(tables_to_refresh):
            table_list = [table.strip() for table in tables_to_refresh.split(",")]
            if not all(table for table in table_list):
                logger.error("Invalid tables_to_refresh parameter.")
                raise ValueError("Invalid tables_to_refresh parameter.")

        # Validate partitions_to_refresh JSON
        if is_valid_text(partitions_to_refresh):
            columns = ["table", "selected_partitions"]
            validate_json(partitions_to_refresh, columns)
    
    # Validate enable_referential_integrity
    if not isinstance(enable_referential_integrity, bool):
        enable_referential_integrity = True

    # Validate refresh mode
    if is_valid_text(refresh_mode):
        if refresh_mode not in AVAILABLE_REFRESH_MODES:
            logger.error(f"Invalid refresh_mode parameter. Available modes: {AVAILABLE_REFRESH_MODES}")
            raise ValueError(f"Invalid refresh_mode parameter. Available modes: {AVAILABLE_REFRESH_MODES}")
    else:
        refresh_type = DEFAULT_REFRESH_TYPE

    # Validate refresh type
    if is_valid_text(refresh_type):
        if refresh_type not in AVAILABLE_REFRESH_TYPES:
            logger.error(f"Invalid refresh_type parameter. Available types: {AVAILABLE_REFRESH_TYPES}")
            raise ValueError(f"Invalid refresh_type parameter. Available types: {AVAILABLE_REFRESH_TYPES}")
    else:
        refresh_type = DEFAULT_REFRESH_TYPE

    # Validate commit mode
    if is_valid_text(refresh_commit_mode):
        if refresh_commit_mode not in AVAILABLE_COMMIT_MODES:
            logger.error(f"Invalid refresh_commit_mode parameter. Available commit modes: {AVAILABLE_COMMIT_MODES}")
            raise ValueError(f"Invalid refresh_commit_mode parameter. Available commit modes: {AVAILABLE_COMMIT_MODES}")
    else:
        refresh_commit_mode = DEFAULT_REFRESH_COMMIT_MODE
    
    # Validate max_parallelism
    if refresh_max_parallelism is None:
        refresh_max_parallelism = DEFAULT_REFRESH_MAX_PARALLELISM
    elif not isinstance(refresh_max_parallelism, int) or refresh_max_parallelism <= 0:
        logger.error("Invalid refresh_max_parallelism parameter.")
        raise ValueError("Invalid refresh_max_parallelism parameter.")
    
    # Validate notebook_timeout
    if notebook_timeout is None:
        notebook_timeout = DEFAULT_NOTEBOOK_TIMEOUT
    elif not isinstance(notebook_timeout, int) or notebook_timeout <= 0:
        logger.error("Invalid notebook_timeout parameter.")
        raise ValueError("Invalid notebook_timeout parameter.")
    
    return {
        "workspace_id": workspace_id,
        "dataset_id": dataset_id,
        "enable_partition": enable_partition,
        "partitions_config": partitions_config,
        "enable_refresh": enable_refresh,
        "tables_to_refresh": tables_to_refresh,
        "partitions_to_refresh": partitions_to_refresh,
        "enable_referential_integrity": enable_referential_integrity,
        "refresh_mode": refresh_mode,
        "refresh_type": refresh_type,
        "refresh_commit_mode": refresh_commit_mode,
        "refresh_max_parallelism": refresh_max_parallelism,
        "notebook_timeout": notebook_timeout
    }

In [21]:
def generate_partitions_list(partitions_config: pd.DataFrame) -> str:
    """
    Generates a JSON-formatted string representing partition ranges for each table in the input DataFrame.

    Args:
        partitions_config (pd.DataFrame): DataFrame containing the partitions configuration.

    Returns:
        str: JSON-formatted string with partitions separated by commas for each table.

    Raises:
        Exception: If any step in the process fails.
    """

    date_ranges = []
    
    for row in partitions_config.itertuples():

        logger.info("Calculating bounds for each table...")
        try:
            first_date: datetime = datetime.strptime(str(row.first_date), DATE_FORMAT)
            refresh_from: datetime = datetime.strptime(str(row.last_date), DATE_FORMAT)

            start_date, end_date = get_bounds_from_offset(
                first_date,
                refresh_from,
                row.interval,
                row.intervals_to_refresh
            )
            
            logger.info("Generating date ranges for each table...")
            date_range: pd.DataFrame = generate_date_ranges(start_date, end_date, row.interval).assign(table=row.table)
            date_ranges.append(date_range)
        except Exception as e:
            logger.error(f"Unable to calculate bounds for partitions: {str(e)}")
            raise

    partitions = pd.concat(date_ranges, ignore_index=True)
    logger.info("Composing partition names to refresh...")
    
    try:
        # Generating partitions list like Table_yyyyMMdd_yyyyMM_dd
        partitions["range_start"] = pd.to_datetime(partitions["range_start"]).dt.strftime(DATE_FORMAT)
        partitions["range_end"] = pd.to_datetime(partitions["range_end"]).dt.strftime(DATE_FORMAT)
        partitions["partition"] = partitions.apply(
            lambda row: f"{row['table']}_{row['range_start']}_{row['range_end']}", axis=1
        )
        
        partitions_agg: pd.DataFrame = (
            partitions.groupby("table", as_index=False)
            .agg(selected_partitions=("partition", lambda x: ",".join(x)))
        )
        objects = partitions_agg.to_json(orient="records")
        
        logger.info("List of partitions to refresh created successfully.")
        
        return objects
    except Exception as e:
        logger.error(f"Unable to format or aggregate partition data: {str(e)}")
        raise

In [22]:
def run_notebook(notebook_name: str, timeout: int, params: Dict[str, Any]) -> None:
    """
    Runs a Fabric notebook with specified parameters and timeout.

    Args:
        notebook_name (str): Name of the notebook to run.
        timeout (int): Timeout for the refresh execution.
        params (Dict[str, Any]): Dictionary of parameters to pass to the notebook.

    Returns:
        None

    Raises:
        RuntimeError: If notebook execution fails.
    """
    try:
        logger.info(f"Running notebook '{notebook_name}'...")
        notebookutils.notebook.run(notebook_name, timeout, params)
        logger.info(f"Notebook '{notebook_name}' completed successfully.")
    except Exception as e:
        logger.error(f"Failed to execute '{notebook_name}' notebook: {str(e)}")
        raise RuntimeError(f"Notebook execution failed for {notebook_name}: {str(e)}") from e

In [23]:
def run():
    """
    Orchestrates dataset partitioning and refreshing in Power BI.
    
    Raises:
        RuntimeError: If any notebook execution fails
    """

    # Validate input parameters
    params = validate_params(
        workspace_id,
        dataset_id,
        enable_partition,
        partitions_config,
        enable_refresh,
        tables_to_refresh,
        partitions_to_refresh,
        enable_referential_integrity,
        refresh_mode,
        refresh_type,
        refresh_commit_mode,
        refresh_max_parallelism,
        notebook_timeout
    )
    
    # Create partitions if enable_partition flag is enabled
    if params["enable_partition"]:
        
        logger.info("Partition dataset is enabled.")

        if not is_valid_text(params["partitions_config"]):
            logger.error("Partitions configuration is required for partitioning.")
            raise ValueError("Partitions configuration is required for partitioning.")
        
        run_notebook(
            PARTITIONER_NOTEBOOK_NAME,
            params["notebook_timeout"],
            {
                "workspace_id": params["workspace_id"], 
                "dataset_id": params["dataset_id"], 
                "partitions_config": 
                params["partitions_config"]
            }
        )
    else:
        logger.info("Partition creation is disabled.")
    
    # Refresh dataset if enable_refresh flag is enabled
    if params["enable_refresh"]:
        
        logger.info("Refresh dataset is enabled.")
        objects: Optional[str] = None

        # Check for explicit refresh configuration
        if is_valid_text(params["partitions_to_refresh"]):
            objects = params["partitions_to_refresh"]
            logger.info(f"Using provided list of partitions to refresh: {params['partitions_to_refresh']}")
        # Generate refresh list because refresh configuration not explicitly provided
        elif is_valid_text(params["partitions_config"]):
            try:
                partitions_config_df : pd.DataFrame = pd.read_json(StringIO(params["partitions_config"]))
                logger.info(f"Creating a list of partitions to refresh for tables: {partitions_config_df['table'].tolist()}\n")
                objects = generate_partitions_list(partitions_config_df)
                logger.info(f"Partitions to refresh:\n{objects}\n")
            except Exception as e:
                logger.error(f"Failed to process refresh configuration: {str(e)}")
                raise
        else:
            logger.warning("No refresh information provided. All partitions will be refreshed.")
            
        # Run notebook to refresh dataset
        run_notebook(
            REFRESHER_NOTEBOOK_NAME,
            params["notebook_timeout"],
            {
                "workspace_id": params["workspace_id"],
                "dataset_id": params["dataset_id"],
                "tables_to_refresh": params["tables_to_refresh"],
                "partitions_to_refresh": objects,
                "enable_referential_integrity": params["enable_referential_integrity"],
                "refresh_mode": params["refresh_mode"],
                "refresh_type": params["refresh_type"],
                "commit_mode": params["refresh_commit_mode"],
                "max_parallelism": params["refresh_max_parallelism"]
            }
        )
        logger.info("Dataset refresh completed successfully.")
    else:
        logger.info("Refresh dataset is disabled.")

In [26]:
if __name__ == "__main__":
    run()

[INFO] 16:43:29 - Partition dataset is enabled.
[INFO] 16:43:29 - Running notebook 'NB_PAR_PARTITIONER'...


[INFO] 16:44:24 - Notebook 'NB_PAR_PARTITIONER' completed successfully.
[INFO] 16:44:24 - Refresh dataset is enabled.
[INFO] 16:44:24 - Creating a list of partitions to refresh for tables: ['Sales']

[INFO] 16:44:24 - Calculating bounds for each table...
[INFO] 16:44:24 - Generating date ranges for each table...
[INFO] 16:44:25 - Composing partition names to refresh...
[INFO] 16:44:25 - List of partitions to refresh created successfully.
[INFO] 16:44:25 - Partitions to refresh:
[{"table":"Sales","selected_partitions":"Sales_20200101_20200331,Sales_20200401_20200630,Sales_20200701_20200930,Sales_20201001_20201231,Sales_20210101_20210331,Sales_20210401_20210630,Sales_20210701_20210930,Sales_20211001_20211231,Sales_20220101_20220331,Sales_20220401_20220630,Sales_20220701_20220930,Sales_20221001_20221231,Sales_20230101_20230331,Sales_20230401_20230630,Sales_20230701_20230930,Sales_20231001_20231231,Sales_20240101_20240331,Sales_20240401_20240630,Sales_20240701_20240930,Sales_20241001_20241